# DD Startup Analysis - Run Interface

This notebook provides a simple interface to run DD startup simulations without using the command line.

**Equivalent to running:**
```bash
python -m ddstartup <param_file> <config_file>
```

## Quick Start
1. Choose your parameter file and configuration in the cell below
2. Run all cells to execute the analysis
3. Results will be saved to the `outputs/` directory

## Configuration

**Select your files here:**

In [1]:
# ============================================================================
# CHOOSE YOUR FILES HERE
# ============================================================================

# Parameter file (in inputs/ directory)
# Options: 'params', 'params_test', 'params_noPaux', '6params', etc.
PARAM_FILE = 'params_noPaux'  # Can use .yaml or .py extension (optional)

# Configuration file (in inputs/ directory)
# Options: 'parametric_tseeded', 'parametric_lump', 'sobol_tseeded', 'sobol_lump'
CONFIG_FILE = 'parametric_tseeded'  # Can use .yaml extension (optional)

# Additional options
VERBOSE = True      # Show detailed progress
DRY_RUN = False     # Only validate, don't run (set to True for testing)

print(f"Selected configuration:")
print(f"  Parameter file: {PARAM_FILE}")
print(f"  Config file: {CONFIG_FILE}")
print(f"  Verbose: {VERBOSE}")
print(f"  Dry run: {DRY_RUN}")

Selected configuration:
  Parameter file: params_noPaux
  Config file: parametric_tseeded
  Verbose: True
  Dry run: False


## Run Analysis

This cell runs `ddstartup.main()` with your selected configuration.

In [ ]:
import sys
import warnings

# Suppress scipy integration warnings
warnings.filterwarnings("ignore", module="scipy.integrate")

# Import ddstartup
from ddstartup.main import main

def run_ddstartup(param_file, config_file, verbose=True, dry_run=False):
    """
    Run ddstartup.main with specified parameters.
    
    This is equivalent to running:
    python -m ddstartup <param_file> <config_file> [--verbose] [--dry-run]
    """
    # Save original argv
    original_argv = sys.argv.copy()
    
    try:
        # Set up arguments
        sys.argv = ['ddstartup', param_file, config_file]
        if verbose:
            sys.argv.append('--verbose')
        if dry_run:
            sys.argv.append('--dry-run')
        
        print(f"\nRunning: ddstartup {param_file} {config_file}" + 
              (" --verbose" if verbose else "") + 
              (" --dry-run" if dry_run else ""))
        print("=" * 80)
        
        # Run main
        result = main()
        
        print("=" * 80)
        if result == 0:
            print("✅ Analysis completed successfully!")
        else:
            print("❌ Analysis failed with errors.")
        
        return result
        
    finally:
        # Restore original argv
        sys.argv = original_argv

# Run the analysis
exit_code = run_ddstartup(
    param_file=PARAM_FILE,
    config_file=CONFIG_FILE,
    verbose=VERBOSE,
    dry_run=DRY_RUN
)


Running: ddstartup params_noPaux parametric_tseeded --verbose

SYSTEM PROFILE
Hardware:
  CPU Cores: 12
  Total RAM: 8.2 GB
  Available RAM: 5.0 GB (61.5% free)

Recommended Parallel Processing Parameters:
  n_jobs: 11 (parallel workers)
  chunk_size: 5500 (computations per chunk)
  batch_size: 250 (results buffer size)


DD STARTUP ANALYSIS CONFIGURATION
Parameter file: inputs/params_noPaux.yaml
Config file: inputs/parametric_tseeded.yaml
Analysis type: T_seeded
Method: parametric
Max simulation time: 10.00 years
Vector length: 100
n_jobs: 11
chunk_size: 5500
batch_size: 250
Output directory: outputs

Input parameter fields:
  V_plasma            : shape=(5,), range=[1.000e+02, 1.500e+03]
  T_i                 : shape=(3,), range=[1.000e+01, 2.000e+01]
  n_tot               : shape=(3,), range=[8.000e+19, 2.000e+20]
  tau_p_T             : shape=(4,), range=[1.000e-01, 2.000e+00]
  P_aux               : shape=(1,), range=[nan, nan]
  P_aux_DT_eq         : shape=(1,), range=[nan, nan]

🔄 COMPUTE (write: 0.48s, ✓ 0.0%): :  35%|███▌      | 165500/466560 [03:10<07:08, 703.11comb/s] 

## Postprocessing (Optional)

Generate plots and analyze results using a postprocessing configuration file.

In [10]:
# ============================================================================
# POSTPROCESSING CONFIGURATION
# ============================================================================

# Postprocess configuration file (in inputs/ directory)
# Options: 'postprocess_config', 'postprocess_quick'
POSTPROCESS_CONFIG = 'postprocess_config'  # Can use .yaml extension (optional)

# Set to True to run postprocessing automatically after analysis
RUN_POSTPROCESSING = True

print(f"Postprocessing configuration:")
print(f"  Config file: {POSTPROCESS_CONFIG}")
print(f"  Auto-run: {RUN_POSTPROCESSING}")

# ============================================================================
# RUN POSTPROCESSING
# ============================================================================

if RUN_POSTPROCESSING and exit_code == 0:
    print("\n" + "=" * 80)
    print("STARTING POSTPROCESSING")
    print("=" * 80)
    
    # Import postprocessing CLI
    from ddstartup.postprocessing.cli import main as postprocess_main
    
    # Save original argv
    original_argv = sys.argv.copy()
    
    try:
        # Set up arguments for postprocessing
        sys.argv = ['ddstartup.postprocessing', POSTPROCESS_CONFIG]
        
        print(f"\nRunning: ddstartup.postprocessing {POSTPROCESS_CONFIG}")
        print("=" * 80)
        
        # Run postprocessing
        postprocess_result = postprocess_main()
        
        print("=" * 80)
        if postprocess_result == 0:
            print("✅ Postprocessing completed successfully!")
        else:
            print("❌ Postprocessing failed with errors.")
            
    finally:
        # Restore original argv
        sys.argv = original_argv
        
elif RUN_POSTPROCESSING and exit_code != 0:
    print("\n⚠️ Skipping postprocessing because analysis failed.")
else:
    print("\nℹ️ Postprocessing disabled. Set RUN_POSTPROCESSING = True to enable.")

/home/alessmor/Scrivania/dd_startup/ddstartup/postprocessing/cli.py:52: UserWarning: hdf5plugin not installed - LZ4 compressed files cannot be read
  warnings.warn("hdf5plugin not installed - LZ4 compressed files cannot be read")


Postprocessing configuration:
  Config file: postprocess_config
  Auto-run: True

STARTING POSTPROCESSING

Running: ddstartup.postprocessing postprocess_config
📋 Loading configuration from: postprocess_config.yaml
📂 Using latest folder: 20251104_161351_parametric_T_seeded
   Found 1 HDF5 file(s)

🎯 Target variables: unrealized_profits, t_startup
📊 Plot types: kde, parcoords, pdf, importance, kmeans, contour, shap
🔷 SHAP interpolation: ENABLED (smooth density plots)
🔷 PDF smoothing: ENABLED (KDE)
💾 Output directory: /home/alessmor/Scrivania/dd_startup/outputs/20251104_161351_parametric_T_seeded (latest folder)

GENERATING PLOTS

📁 Processing: ddstartup_20251104_161351_parametric_T_seeded.h5

  🎯 Target: unrealized_profits
   Loading data (inputs + unrealized_profits)...
   Filtered: 8192 → 6816 rows (83.2%)

Running: ddstartup.postprocessing postprocess_config
📋 Loading configuration from: postprocess_config.yaml
📂 Using latest folder: 20251104_161351_parametric_T_seeded
   Found 1 HDF5